# Biomass LA2 product Congo basin vs CCI Biomass

Objetive: Establish an area of analisis based on the valid pixel available from the AGB LA2 map and the CCI biomass products footprint

### 1. Packages and Libraries

In [ ]:
%pip install aiohttp requests

from shapely.geometry import Polygon, box
from rasterio.windows import Window
import xml.etree.ElementTree as ET
import matplotlib.pyplot as plt
from pystac_client import Client
from folium import GeoJson
import geopandas as gpd
from io import BytesIO
import rasterio as rio
from tqdm import tqdm
from PIL import Image
import pandas as pd
import numpy as np
import requests
import pathlib
import fsspec
import folium
import fiona
import os

fiona.supported_drivers["KML"] = "rw"  # Enable kml driver

from rasterio.windows import from_bounds
from rasterio.plot import show
from rasterio.plot import plotting_extent

## 2. Open the DF with the CCI Biomass information available

In [ ]:
path_gdf = r'CCI_biomass_products_Congo.shp'
gdf = gpd.read_file(path_gdf)

## 3. Open AGB LA2 Congo map

In [ ]:
# Path to your GeoTIFF file
tif_path = r'Examples_and_Data/Workshops/BIOMASS-Coding2026/Biomass_AGBD_Congo_Basin/agbd_map_L2Afullextent_congo_basin.tif'

with rio.open(tif_path) as src:
    print("File:", src.name)
    print("Raster dimensions:", src.width, "x", src.height)
    print("Number of bands:", src.count)
    print("Coordinate reference system:", src.crs)
    print("Pixel resolution:", src.res)
    print("No-data value:", src.nodata)
    print("Band descriptions:", src.descriptions)
    print("Bounds:", src.bounds)

agb_bbox = [coord for coord in src.bounds]

## 4. Check the valid pixels for each CCI Biomass product footprint

In [ ]:
for i, row in gdf.iterrows():
    prod_id = row['product_id']
    minx, miny, maxx, maxy = row['geometry'].bounds
    print(i, prod_id)
    # Replace these example coordinates with your desired zone
    left = minx
    bottom = miny
    right = maxx
    top = maxy
    with rio.open(tif_path) as src:
        # Convert map coordinates into a pixel window
        window = from_bounds(
            left, bottom, right, top,
            transform=src.transform
        )
        # Align the window to whole pixels
        window = window.round_offsets().round_lengths()
        # Crop the raster
        cropped = src.read(1, window=window, masked=True)
        # Update the spatial transform for the cropped raster
        cropped_transform = src.window_transform(window)
        print("Original size:", src.width, "x", src.height)
        print("Cropped size:", cropped.shape[1], "x", cropped.shape[0])
        # Calculate the correct map extent for the cropped zone
        extent = plotting_extent(cropped, cropped_transform)
        
        # Convert masked pixels to NaN for display
        cropped_data = cropped.filled(np.nan)
        
        valid = cropped_data[np.isfinite(cropped_data)]
        valid_pxs = valid.size
        gdf.at[i,'Valid_pixels'] = valid_pxs
        # if valid.size > 0:
        if valid_pxs > 0:    
            #raise ValueError("The selected zone contains no valid pixels.")
        
            vmin, vmax = np.percentile(valid, [2, 98])
            # vmin, vmax = np.nanmin(valid), np.nanmax(valid)
            fig, ax = plt.subplots(figsize=(10, 8))
            # gpd.GeoSeries([row['geometry']]).plot(
            #     ax=ax, 
            #     color='blue', 
            #     edgecolor='black', 
            #     alpha=0.5
            # )
            im = ax.imshow(
                cropped_data,
                extent=extent,
                origin="upper",
                cmap="YlGn",
                vmin=vmin,
                vmax=vmax
            )
            
            cbar = plt.colorbar(im, ax=ax)
            cbar.set_label("AGB (ton/ha)")
            
            ax.set_title(f"BIOMASS Level 2A\nCropped to CCI Biomass product:\n{prod_id}\nValid pixels: {valid_pxs}")
            ax.set_xlabel("X coordinate")
            ax.set_ylabel("Y coordinate")
            
            plt.tight_layout()
            plt.show()
        else: 
            print(f'The selected zone contains no valid pixels\nCCI Biomass Product ID: {prod_id}')

## 5. Select the AOI based on the maximum numer of valid pixels

In [ ]:
# Get the area with the most valid pixels
idx_max_px = np.argmax(gdf['Valid_pixels'])
prod_id = gdf.loc[idx_max_px, 'product_id']
item = gdf.loc[idx_max_px, 'item']
minx, miny, maxx, maxy = gdf.loc[idx_max_px,'geometry'].bounds
print(prod_id, item)
minx, miny, maxx, maxy

## 6. Crop the AGB LA2 Congo map based on the AOI

In [ ]:
# Replace these example coordinates with your desired zone
left = minx
bottom = miny
right = maxx
top = maxy

with rio.open(tif_path) as src:
    meta = src.meta
    # Convert map coordinates into a pixel window
    window = from_bounds(
        left, bottom, right, top,
        transform=src.transform
    )
    # Align the window to whole pixels
    window = window.round_offsets().round_lengths()
    # Crop the raster
    cropped = src.read(1, window=window, masked=True)
    # Update the spatial transform for the cropped raster
    cropped_transform = src.window_transform(window)
    print("Original size:", src.width, "x", src.height)
    print("Cropped size:", cropped.shape[1], "x", cropped.shape[0])
    # Calculate the correct map extent for the cropped zone
    extent = plotting_extent(cropped, cropped_transform)
    
    # Convert masked pixels to NaN for display
    cropped_data = cropped.filled(np.nan)

In [ ]:
out_meta = src.meta
cropped_data_nx = np.expand_dims(cropped_data, axis=0)
cropped_data_nx.shape

In [ ]:
out_meta.update({"driver": "GTiff",
                 "height": cropped_data_nx.shape[1],
                 "width": cropped_data_nx.shape[2],
                 "transform": cropped_transform,
                 "count":1})

with rio.open(f"agbd_map_L2A_cropped_{prod_id}.tif", "w", **out_meta) as dest:
    dest.write(cropped_data_nx)

In [ ]:
# Save gdf with the valid pixels ifno
gdf.to_csv('CCI_biomass_products_Congo_valid_pxs.csv', index=False)